# MTR Demo: Scene Generator & Observation Simulator
<!-- ![alt text](/home/gonzalezm/test/test2/s2gos-apps/example/s2gos-logo.png) -->
<img src="/home/gonzalezm/test/test2/s2gos-apps/example/s2gos-logo.png" width="20%">

---
This notebook demonstrates the **Scene Generator** and **Observation Simulator** operating within the S2GOS concept.

We will simulate a scene in **Patagonia National Park (PNP)** to showcase:
*   General scene generation.
*   Seasonal variations (vegetation, lighting, atmosphere).
*   Preparing a theoretical installation of a HYPSTAR instrument
*   Satellite and ground observation as well as theoretical measurements.


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.gridspec import GridSpec

## Scene Generation
In DTE-S2GOS scene genration is composed of several steps (some optional) and pulling in various data to assemble a Scene Description file and realated assets. We generate a DAG of this various steps to better handle the complexity and open the door to distributing the generation in the future.

<img src="/home/gonzalezm/test/test2/s2gos-apps/example/mtr_28_1_june/gen_output/pnp_mtr_demo_june_seed13/pnp_mtr_demo_june_seed13_dag.png">

Althrough we are concentrated in five ROIs where we will try to pull in as much site specific data as possible, the Scene Generator is built with basic global coverage in mind. At the moment we are accessing WorldCover on OVH cloud set up by Brockmann Consulting and the Copernicus DEM through Earth Data Hub we also use a variety of materials many coming from the ECOSTRESS library from NASA. Today we will be focusing on the PNP site for the land upscaling use case and we will use trees coming from TLS scans gathered and processed by the University of Ghent. We will generate the scene using 21/01/2021 () as our reference date.

In [ ]:
# Running scene gen

In [ ]:
# Inspect

Here is a prerecorded video showing the current scene as well as the effect of seasonality if you chose to generate the scene for December (summer in Patagonia) instead.

In [ ]:
from IPython.display import HTML

HTML("""
<video width="1280" controls autoplay loop>
  <source src="/home/gonzalezm/test/test2/s2gos-apps/example/mtr_rgb_viz_june/sim_output/flyby_combined_season_morph_h264.mp4" type="video/mp4">
</video>
""")

In [ ]:
# Running sim

## Results
Due to the time needed to run simulations we have prepared results showcasing early results. 

### CHIME

The **Copernicus Hyperspectral Imaging Mission for the Environment (CHIME)** is a Sentinel expansion mission designed to support sustainable agriculture, biodiversity, and soil management. Its key instrument is a **Hyperspectral Imager** that captures the scene in over 200 continuous spectral bands (400–2500 nm) at a 30m ground resolution, allowing for precise material identification. CHIME-A has a working launch of 2028 but with our Observation Simulator built on the **Eradiate** radiative transfer model we can start simulating it over over regions of interest.


<img src="/home/gonzalezm/test/test2/s2gos-apps/experimenting/CHIME_satellite.png" >


In [ ]:
chime_december = xr.load_dataset(
    "/home/gonzalezm/test/test2/s2gos-apps/example/MTR_precalculated_results/mtr_demo_chime_hsi_december.zarr"
)

In [ ]:
chime_december

In [ ]:
# Calculate TOA BRF (Bidirectional Reflectance Factor)
# BRF = (π * L) / (E * cos(θ_s))

radiance = chime_december.radiance
irradiance = chime_december.toa_irradiance

sza = chime_december.sza  # in degrees
cos_sza = np.cos(np.radians(sza))

brf = (np.pi * radiance) / (irradiance * cos_sza)

chime_december["brf"] = brf
chime_december["brf"].attrs.update(
    {
        "long_name": "Bidirectional Reflectance Factor",
        "standard_name": "toa_bidirectional_reflectance",
        "units": "dimensionless",
        "description": "TOA BRF calculated as π × L / E × cos(θ_s)",
        "formula": "pi * radiance * cos(sza) / toa_irradiance",
    }
)

In [ ]:
from IPython.display import HTML

# --- PREP DATA ---
chime_december = chime_december["brf"].squeeze()
chime_december = chime_december.isel(w=slice(1, None))
chime_december = chime_december.isel(x_index=slice(1, -1), y_index=slice(1, -1))
chime_december = chime_december.isel(y_index=slice(None, None, -1))
chime_december = chime_december.fillna(0)
mean_spectrum = chime_december.mean(dim=["x_index", "y_index"])

vmin = 0
vmax = chime_december.quantile(0.99).item()

# --- FIGURE ---
fig = plt.figure(figsize=(8, 10), facecolor="black")
gs = GridSpec(4, 1, figure=fig)

# Top: Image
ax_img = fig.add_subplot(gs[0:3, :])
ax_img.axis("off")
ax_img.set_title("CHIME TOA BRF", color="white", fontsize=16)
im = ax_img.imshow(chime_december[0, :, :], vmin=vmin, vmax=vmax, cmap="gray")
cbar = plt.colorbar(im, ax=ax_img, fraction=0.046, pad=0.04)
cbar.ax.tick_params(color="white", labelcolor="white")
cbar.set_label("TOA BRF", color="white")

# Bottom: Spectrum
ax_spec = fig.add_subplot(gs[3, :])
ax_spec.set_facecolor("#111111")
ax_spec.spines["bottom"].set_color("white")
ax_spec.spines["left"].set_color("white")
ax_spec.tick_params(axis="x", colors="white")
ax_spec.tick_params(axis="y", colors="white")
ax_spec.set_xlabel("Wavelength (nm)", color="white")
ax_spec.set_ylabel("Mean TOA BRF", color="white")

ax_spec.plot(chime_december.w, mean_spectrum, color="gray", alpha=0.6)
line_indicator = ax_spec.axvline(
    x=chime_december.w[0], color="cyan", linestyle="--", alpha=0.8
)
(dot_indicator,) = ax_spec.plot(
    chime_december.w[0], mean_spectrum[0], "o", color="cyan", markersize=8
)

wl_text = ax_img.text(
    0.02,
    0.95,
    "",
    transform=ax_img.transAxes,
    color="cyan",
    fontsize=14,
    fontweight="bold",
)


# --- UPDATE FUNCTION ---
def update(frame_idx):
    im.set_data(chime_december[frame_idx, :, :])
    current_w = chime_december.w[frame_idx].values
    current_val = mean_spectrum[frame_idx].values
    wl_text.set_text(f"λ: {current_w:.1f} nm")
    line_indicator.set_xdata([current_w])
    dot_indicator.set_data([current_w], [current_val])
    return im, wl_text, line_indicator, dot_indicator


# --- ANIMATION ---
ani = FuncAnimation(fig, update, frames=len(chime_december.w), interval=50, blit=False)

# --- EMBED INLINE IN NOTEBOOK ---
import matplotlib as mpl

mpl.rcParams["animation.embed_limit"] = 200  # ensure enough space for hyperspectral

plt.close(fig)  # prevent static duplicate
HTML(ani.to_jshtml())

## HYPSTAR-XR
We are also working on simulating the HYPSTAR-XR sensor, which ahs been used extensively as part of the LANDHYPERNET project including in Gobabeb (one of our ROIs) and in forested areas such as Wytham Woods. HYPSTAR-XR cover 380–1680 nm at 10nm spectra resolution with a 5 degree circular FOV, here we insert a virtual tower in the style of Wytham Woods and simulate the results a HYPSTAR-XR instrument could gather in this environment


<img src="/home/gonzalezm/test/test2/s2gos-apps/example/MTR_precalculated_results/hypstar_wytham.png" width="20%">


As part of the simulation we can generate a preview of the HYPSTAR-XR for some verification

<img src="/home/gonzalezm/test/test2/s2gos-apps/example/MTR_precalculated_results/FOV.png" width="20%">

<img src="/home/gonzalezm/test/test2/s2gos-apps/example/MTR_precalculated_results/hypstar_rgb_december.png" width="20%">


In [ ]:
hypstar_december = xr.load_dataset(
    "/home/gonzalezm/test/test2/s2gos-apps/example/MTR_precalculated_results/mtr_hypstar_hcrf_december.zarr"
)

In [ ]:
plt.figure(figsize=(10, 8))

# Plot Wavelength (w) vs HDRF
plt.plot(hypstar_december.w, hypstar_december.hdrf, color="#1f77b4", linewidth=1.5)

# Labels and Title
plt.ylabel("HCRF", fontsize=12)
plt.xlabel("Wavelength (nm)", fontsize=12)
plt.title("HYPSTAR-XR HCRF PNP (VZA=3.0°)", fontsize=14, fontweight="bold")

# Styling options for a cleaner scientific look
plt.grid(True, linestyle="--", alpha=0.6)
plt.xlim(
    hypstar_december.w.min(), hypstar_december.w.max()
)  # Tighten x-axis to data range
plt.ylim(0, 0.6)  # Start Y at 0 and add 5% headroom at top
plt.savefig("hypstar_hcrf_june.png", dpi=300, bbox_inches="tight", facecolor="white")

plt.tight_layout()
plt.show()

Although this results are very early and it is not something that we can really validate given there is no actual tower at the site we see that HCRF shape resembles what was gathered from the Wytham woods canopy:

<img src="/home/gonzalezm/test/test2/s2gos-apps/example/MTR_precalculated_results/wytham_woods_real_HCRF.png" width="20%">

## Pixel Level 2 Surface Reflectance (HDRF)

In [ ]:
tower_pixel_value = xr.load_dataset(
    "/home/gonzalezm/test/test2/s2gos-apps/example/MTR_precalculated_results/mtr_demo_satellite_pixel_hdrf_chime_pixel_HDRF_r138_c159.zarr"
)

In [ ]:
plt.figure(figsize=(10, 8))

# Plot Wavelength (w) vs HDRF
plt.plot(
    tower_pixel_value.w.squeeze(),
    tower_pixel_value.hdrf.squeeze(),
    color="#1f77b4",
    linewidth=1.5,
)

# Labels and Title
plt.ylabel("HDRF", fontsize=12)
plt.xlabel("Wavelength (nm)", fontsize=12)
plt.title("CHIME Pixel HDRF at tower location", fontsize=14, fontweight="bold")

# Styling options for a cleaner scientific look
plt.grid(True, linestyle="--", alpha=0.6)
plt.xlim(
    tower_pixel_value.w.min(), tower_pixel_value.w.max()
)  # Tighten x-axis to data range
plt.ylim(0, 0.6)  # Start Y at 0 and add 5% headroom at top
# plt.savefig("hypstar_hcrf_june.png", dpi=300, bbox_inches='tight', facecolor='white')

plt.tight_layout()
plt.show()